# **03_analisis_series_tiempo**

Este notebook desarrolla el análisis formal de series temporales sobre la variable objetivo
`target_kwh`, con énfasis en estacionariedad, diferenciación e identificación preliminar
de modelos ARIMA.

## Objetivos
- Evaluar la estacionariedad de la serie original mediante la prueba ADF.
- Analizar visualmente la necesidad de diferenciación.
- Aplicar diferenciación de primer orden.
- Verificar la estacionariedad de la serie diferenciada.
- Analizar ACF y PACF para proponer órdenes ARIMA candidatos.
- Generar evidencia técnica para el Capítulo 4 del TFM.

**BLOQUE 2 — Librerías y configuración**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

**BLOQUE 3 — Rutas**

In [ ]:
# ==========================================
# BLOQUE 3 — Configuración de rutas
# ==========================================

from pathlib import Path

# Detectar si está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"

# Crear carpetas si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

# Archivo de entrada
INPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)

# Validación
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {INPUT_FILE}\n"
        "Verifica que el archivo exista en data/processed/"
    )

**BLOQUE 4 — Carga y preparación de la serie**

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

DATE_COL = "date"
Y_COL = "target_kwh"

if DATE_COL not in df.columns:
    raise ValueError(f"No se encontró la columna '{DATE_COL}'.")

if Y_COL not in df.columns:
    raise ValueError(f"No se encontró la columna '{Y_COL}'.")

df = df.sort_values(DATE_COL).set_index(DATE_COL)

y = df[[Y_COL]].copy()

print("Dimensión de la serie:", y.shape)
print("Frecuencia inferida:", pd.infer_freq(y.index))
display(y.head())
display(y.tail())

**BLOQUE 5 — Prueba ADF sobre la serie original**

In [ ]:
series_original = y[Y_COL].dropna()

adf_result_original = adfuller(series_original, autolag="AIC")

adf_original_summary = pd.DataFrame({
    "metric": [
        "adf_statistic",
        "p_value",
        "used_lags",
        "n_obs",
        "critical_value_1pct",
        "critical_value_5pct",
        "critical_value_10pct"
    ],
    "value": [
        adf_result_original[0],
        adf_result_original[1],
        adf_result_original[2],
        adf_result_original[3],
        adf_result_original[4]["1%"],
        adf_result_original[4]["5%"],
        adf_result_original[4]["10%"]
    ]
})

display(adf_original_summary)
adf_original_summary.to_csv(OUTPUTS_TABLES / "table_adf_original_series.csv", index=False)

**BLOQUE 6 — Interpretación automática ADF original**

In [ ]:
adf_stat = adf_result_original[0]
p_value = adf_result_original[1]
cv_5 = adf_result_original[4]["5%"]

print("Interpretación ADF - serie original")
print(f"ADF statistic: {adf_stat:.6f}")
print(f"p-value: {p_value:.6f}")
print(f"Critical value 5%: {cv_5:.6f}")

if p_value < 0.05 and adf_stat < cv_5:
    print("Resultado: existe evidencia estadística de estacionariedad al 5%.")
else:
    print("Resultado: no hay evidencia suficiente de estacionariedad al 5%.")

**BLOQUE 7 — Visualización de la serie original**

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(y.index, y[Y_COL], linewidth=0.9, label="Serie original")

ax.set_title("Serie original de demanda eléctrica diaria", fontsize=13)
ax.set_xlabel("Fecha", fontsize=11)
ax.set_ylabel("Demanda eléctrica diaria (kWh)", fontsize=11)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_xlim(y.index.min(), y.index.max())

ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "serie_original_demanda.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 8 — Diferenciación de primer orden**

In [ ]:
y_diff_1 = y[Y_COL].diff().dropna().to_frame(name="target_kwh_diff_1")

print("Dimensión de la serie diferenciada:", y_diff_1.shape)
display(y_diff_1.head())
display(y_diff_1.tail())

**BLOQUE 9 — Prueba ADF sobre la serie diferenciada**

In [ ]:
adf_result_diff_1 = adfuller(y_diff_1["target_kwh_diff_1"], autolag="AIC")

adf_diff_summary = pd.DataFrame({
    "metric": [
        "adf_statistic",
        "p_value",
        "used_lags",
        "n_obs",
        "critical_value_1pct",
        "critical_value_5pct",
        "critical_value_10pct"
    ],
    "value": [
        adf_result_diff_1[0],
        adf_result_diff_1[1],
        adf_result_diff_1[2],
        adf_result_diff_1[3],
        adf_result_diff_1[4]["1%"],
        adf_result_diff_1[4]["5%"],
        adf_result_diff_1[4]["10%"]
    ]
})

display(adf_diff_summary)
adf_diff_summary.to_csv(OUTPUTS_FIGURES / "table_adf_differenced_series.csv", index=False)

**BLOQUE 10 — Interpretación automática ADF diferenciada**

In [ ]:
adf_stat_diff = adf_result_diff_1[0]
p_value_diff = adf_result_diff_1[1]
cv_5_diff = adf_result_diff_1[4]["5%"]

print("Interpretación ADF - serie diferenciada")
print(f"ADF statistic: {adf_stat_diff:.6f}")
print(f"p-value: {p_value_diff:.6f}")
print(f"Critical value 5%: {cv_5_diff:.6f}")

if p_value_diff < 0.05 and adf_stat_diff < cv_5_diff:
    print("Resultado: la serie diferenciada presenta evidencia de estacionariedad al 5%.")
else:
    print("Resultado: la serie diferenciada aún no presenta evidencia suficiente de estacionariedad al 5%.")

**BLOQUE 11 — Gráfico de la serie diferenciada**

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(
    y_diff_1.index,
    y_diff_1["target_kwh_diff_1"],
    linewidth=0.9,
    label="Serie diferenciada"
)

ax.axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=0.7,
    label="Media = 0"
)

ax.set_title("Serie de demanda eléctrica diaria tras diferenciación de primer orden", fontsize=13)
ax.set_xlabel("Fecha", fontsize=11)
ax.set_ylabel("Variación diaria de la demanda (kWh)", fontsize=11)

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlim(y_diff_1.index.min(), y_diff_1.index.max())

ax.grid(True, alpha=0.3)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "serie_diferenciada_demanda.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 12 — ACF y PACF de la serie diferenciada**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(
    y_diff_1["target_kwh_diff_1"],
    lags=40,
    ax=axes[0],
    alpha=0.05
)
axes[0].set_title("Función de autocorrelación (ACF) - serie diferenciada", fontsize=12)
axes[0].set_xlabel("Rezago")
axes[0].set_ylabel("Autocorrelación")

plot_pacf(
    y_diff_1["target_kwh_diff_1"],
    lags=40,
    ax=axes[1],
    alpha=0.05,
    method="ywm"
)
axes[1].set_title("Función de autocorrelación parcial (PACF) - serie diferenciada", fontsize=12)
axes[1].set_xlabel("Rezago")
axes[1].set_ylabel("Autocorrelación parcial")

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "acf_pacf_demand.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 13 — Registro de candidatos preliminares**

In [ ]:
arima_candidates = pd.DataFrame({
    "candidate_model": [
        "ARIMA(0,1,1)",
        "ARIMA(1,1,0)",
        "ARIMA(1,1,1)"
    ],
    "justification": [
        "Pico significativo en ACF en rezago 1 y rápida disipación posterior.",
        "Pico significativo en PACF en rezago 1 y decaimiento posterior.",
        "Modelo parsimonioso de comparación que combina componentes AR y MA."
    ]
})

display(arima_candidates)
arima_candidates.to_csv(OUTPUTS_TABLES / "table_arima_candidates_preliminary.csv", index=False)

Aunque la prueba de Dickey-Fuller aumentada (ADF) aplicada a la serie original sugiere estacionariedad desde el punto de vista estadístico, el análisis visual y el comportamiento de la media móvil evidencian la presencia de una tendencia creciente, lo que indica que la serie no es estacionaria en sentido práctico.

Por esta razón, se decidió aplicar una diferenciación de primer orden con el objetivo de eliminar la tendencia y estabilizar la media de la serie. Tras la diferenciación, tanto la inspección visual como la prueba ADF confirman que la serie resultante presenta un comportamiento estacionario.

En consecuencia, se establece un orden de diferenciación d = 1 para la modelación ARIMA.